In [ ]:
!pip install -q -U transformers huggingface_hub hf-transfer

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.


In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_ID = "facebook/nllb-200-distilled-600M"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    src_lang="por_Latn"
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype
).to(device)

model.eval()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

... (6 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
       

In [ ]:
import re
import hashlib
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm

DATASET = "bratao/corpus-ptbr-v1"
MAX_AMOSTRAS = 1000
MIN_PALAVRAS = 50
MAX_PALAVRAS = 75

ds = load_dataset(
    DATASET,
    split="train",
    streaming=True
)

README.md:   0%|          | 0.00/16.9k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [ ]:
amostras = []
vistos = set()

for exemplo in tqdm(ds, desc="Extraindo textos"):
    texto = exemplo["text"].strip()

    # Apenas dados reais
    if exemplo["subset"] != "real":
        continue

    num_palavras = len(texto.split())

    if not MIN_PALAVRAS <= num_palavras <= MAX_PALAVRAS:
        continue

    # Normalização para deduplicação
    normalizado = re.sub(r"\s+", " ", texto).casefold()
    identificador = hashlib.sha256(
        normalizado.encode("utf-8")
    ).hexdigest()

    if identificador in vistos:
        continue

    vistos.add(identificador)

    amostras.append(texto)

    if len(amostras) >= MAX_AMOSTRAS:
        break





Extraindo textos: 0it [00:00, ?it/s]

Extraindo textos: 1it [00:17, 17.68s/it]

Extraindo textos: 384it [00:17, 30.75it/s]

Extraindo textos: 767it [00:17, 73.69it/s]

Extraindo textos: 1134it [00:17, 130.69it/s]

Extraindo textos: 1517it [00:18, 212.73it/s]

Extraindo textos: 1918it [00:18, 329.32it/s]

Extraindo textos: 2296it [00:18, 473.66it/s]

Extraindo textos: 2670it [00:18, 657.12it/s]

Extraindo textos: 3053it [00:18, 893.83it/s]

Extraindo textos: 3428it [00:18, 1168.88it/s]

Extraindo textos: 3806it [00:18, 1484.63it/s]

Extraindo textos: 4192it [00:18, 1833.87it/s]

Extraindo textos: 4571it [00:18, 2171.81it/s]

Extraindo textos: 4949it [00:19, 2460.27it/s]

Extraindo textos: 5320it [00:19, 2676.74it/s]

Extraindo textos: 5686it [00:19, 2906.30it/s]

Extraindo textos: 6055it [00:19, 3101.92it/s]

Extraindo textos: 6444it [00:19, 3309.19it/s]

Extraindo textos: 6833it [00:19, 3466.27it/s]

Extraindo textos: 7210it [00:19, 3526.49it/s]

Extraindo textos: 7584it [00:19, 3517

In [ ]:
with open("amostras.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(amostras))

In [ ]:
texto = amostras[0]
texto

'Arquivos tratamentos - Estação do Corpo Importância de tratamentos em clínicas de recuperação Os tratamentos com os dependentes químicos avançou muito nas últimas décadas. Até meados da década de 1960, a dependência não era vista como uma doença, mas como uma falha de caráter. Como são feitas as perucas naturais? Em diferentes formatos e cores, as perucas são itens indispensáveis para elevar a autoestima de pessoas, que por algum motivo, tenham perdido os fios.'

In [ ]:
import torch
from tqdm.auto import tqdm

def traduzir_batch(textos, idioma, batch_size=32):
    idiomas = {
        "english": "eng_Latn",
        "spanish": "spa_Latn"
    }

    resultados = []
    target_lang = tokenizer.convert_tokens_to_ids(idiomas[idioma])

    model.eval()

    for inicio in tqdm(
        range(0, len(textos), batch_size),
        desc=f"Traduzindo para {idioma}"
    ):
        batch = textos[inicio:inicio + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            # max_length=512
        ).to(model.device)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=target_lang,
                max_new_tokens=256,
                num_beams=4
            )

        traducoes_batch = tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True
        )

        resultados.extend(traducoes_batch)

    return resultados


# Lista original
textos = list(amostras)

# Traduções em batches
ingles = traduzir_batch(
    textos,
    idioma="english",
    batch_size=32
)

espanhol = traduzir_batch(
    textos,
    idioma="spanish",
    batch_size=32
)

# Resultado final: lista de dicionários
traducoes = [
    {
        "portuguese": pt,
        "english": en,
        "spanish": es
    }
    for pt, en, es in zip(textos, ingles, espanhol)
]

print(f"Total de textos traduzidos: {len(traducoes)}")
print(traducoes[0])


Traduzindo para english:   0%|          | 0/32 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Traduzindo para spanish:   0%|          | 0/32 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Total de textos traduzidos: 1000
{'portuguese': 'Arquivos tratamentos - Estação do Corpo Importância de tratamentos em clínicas de recuperação Os tratamentos com os dependentes químicos avançou muito nas últimas décadas. Até meados da década de 1960, a dependência não era vista como uma doença, mas como uma falha de caráter. Como são feitas as perucas naturais? Em diferentes formatos e cores, as perucas são itens indispensáveis para elevar a autoestima de pessoas, que por algum motivo, tenham perdido os fios.', 'english': 'Treatments Archives - Body Station Importance of treatments in recovery clinics Treatments with chemical addicts have made great strides in recent decades. Until the mid-1960s, addiction was not seen as a disease, but as a failure of character. How are natural wigs made? In different shapes and colors, wigs are indispensable items for raising the self-esteem of people who for some reason have lost their wires.', 'spanish': 'Los tratamientos con dependientes químicos 

In [ ]:
import json

with open("traducoes.json", "w", encoding="utf-8") as f:
    json.dump(traducoes, f, ensure_ascii=False, indent=4)

print(f"Arquivo salvo com {len(traducoes)} traduções!")

Arquivo salvo com 1000 traduções!
